In [35]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

In [36]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import SystemMessage, HumanMessage

agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3.8-27b",
            trigger=("messages", 10),
            keep=("messages", 4)
        )
    ],
)

In [37]:
## Run with thread id
config = {"configurable": {"thread_id": "test_1"}}

In [38]:
question = [
    "What is the capital of France?",
    "What is the population of France?",
    "What is the currency of France?",
    "What is 2+2?",
    "What is the capital of Germany?",
    "What is the population of Germany?",
    "What is the currency of Germany?",
    "What is 3+3?",
]

for q in question:
    response = agent.invoke({"messages": [HumanMessage(content=q)]}, config=config)
    print(f"Message: {response}")
    print(f"Messages: {len(response['messages'])}")

Message: {'messages': [HumanMessage(content='What is the capital of France?', additional_kwargs={}, response_metadata={}, id='8d860d8a-979a-4f31-a67c-7b8420e1f732'), AIMessage(content='The capital of France is **Paris**.', additional_kwargs={'reasoning_content': 'The user asks a simple question: "What is the capital of France?" The answer: Paris. Provide answer.'}, response_metadata={'token_usage': {'completion_tokens': 42, 'prompt_tokens': 78, 'total_tokens': 120, 'completion_time': 0.088970867, 'completion_tokens_details': {'reasoning_tokens': 24}, 'prompt_time': 0.003094905, 'prompt_tokens_details': None, 'queue_time': 0.309068021, 'total_time': 0.092065772}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_5082008e34', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a09a59-4933-73e0-8c0d-b204077bb2ef-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 78, 'output_tokens': 42, 'total

Token Size

In [39]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import SystemMessage, HumanMessage

@tool
def search_hotels(city: str) -> str:
    """Search for hotels in a given city and return a list of hotel names."""
    return f"""Found hotels in {city}:
    Hotel A $450, Hotel B $300, Hotel C $600"""

agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:openai/gpt-oss-120b",
            trigger=("tokens", 550),
            keep=("tokens", 200)
        )
    ], 
)

config = {"configurable": {"thread_id": "test_2"}}

def count_tokens(messages):
    return sum(len(m.content.split()) for m in messages)

In [40]:
cities = ["New York", "Los Angeles", "Chicago", "Houston", "Phoenix"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Search for hotels in {city}.")]},
        config=config
    )

    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"Message: {response['messages']}")

New York: ~83 tokens, 4 messages
Message: [HumanMessage(content='Search for hotels in New York.', additional_kwargs={}, response_metadata={}, id='dae2ba69-eb3b-4095-9c91-b3c73ded4622'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants to search for hotels in New York. Use the provided function search_hotels.', 'tool_calls': [{'id': 'fc_71cd07e8-ab0b-4b71-b0ac-992087db6643', 'function': {'arguments': '{"city":"New York"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 136, 'total_tokens': 185, 'completion_time': 0.102142142, 'completion_tokens_details': {'reasoning_tokens': 20}, 'prompt_time': 0.005523372, 'prompt_tokens_details': None, 'queue_time': 0.346507767, 'total_time': 0.107665514}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_6b677c2caf', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run-

Fractions

In [41]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import SystemMessage, HumanMessage

@tool
def search_hotels(city: str) -> str:
    """Search for hotels in a given city and return a list of hotel names."""
    return f"""Found hotels in {city}:
    Hotel A $450, Hotel B $300, Hotel C $600"""

agent = create_agent(
    model="groq:openai/gpt-oss-20b",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:openai/gpt-oss-20b",
            trigger=("fraction", 0.005),
            keep=("fraction", 0.002)
        )
    ], 
)

config = {"configurable": {"thread_id": "test_2"}}

def count_tokens(messages):
    return sum(len(m.content.split()) for m in messages)

In [42]:
cities = ["New York", "Los Angeles", "Chicago", "Houston", "Phoenix"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Search for hotels in {city}.")]},
        config=config
    )

    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"Message: {response['messages']}")

New York: ~72 tokens, 4 messages
Message: [HumanMessage(content='Search for hotels in New York.', additional_kwargs={}, response_metadata={}, id='ac9e0175-a68d-4aba-8e00-202d56bfae82'), AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call the function search_hotels with city "New York".', 'tool_calls': [{'id': 'fc_a31a2dbb-979f-4ae2-ab64-22aa855f12e6', 'function': {'arguments': '{"city":"New York"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 136, 'total_tokens': 177, 'completion_time': 0.044675672, 'completion_tokens_details': {'reasoning_tokens': 16}, 'prompt_time': 0.007665778, 'prompt_tokens_details': None, 'queue_time': 0.345342928, 'total_time': 0.05234145}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_3023a70d60', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a09a5a-45bd-7aa1-9035-e2

Human Approval

In [43]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import SystemMessage, HumanMessage

def read_email_tool(email_id: str) -> str:
    """Read the content of an email given its ID."""
    # Simulate reading an email
    return f"Email content for ID {email_id}: This is a sample email content."

def send_email_tool(email_id: str, subject: str, body: str) -> str:
    """Send an email with the given subject and body to the specified email ID."""
    # Simulate sending an email
    return f"Email sent to {email_id} with subject '{subject}' and body '{body}'."

In [44]:
agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False
            }
        )
    ]      
)

In [45]:
config = {"configurable": {"thread_id": "test_3"}}

result = agent.invoke({
    "messages": [HumanMessage(content="Send an email to test@example.com with subject 'Meeting' and body 'Let's meet tomorrow at 10 AM.'")],}, 
    config=config
)

In [46]:
result

{'messages': [HumanMessage(content="Send an email to test@example.com with subject 'Meeting' and body 'Let's meet tomorrow at 10 AM.'", additional_kwargs={}, response_metadata={}, id='aa1a6412-f4df-4936-93f0-a411a61591eb'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to send email using send_email_tool. Provide email_id, subject, body.', 'tool_calls': [{'id': 'fc_9e0b59be-8959-44e8-8dd8-8848d094d858', 'function': {'arguments': '{"body":"Let\'s meet tomorrow at 10 AM.","email_id":"test@example.com","subject":"Meeting"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 70, 'prompt_tokens': 188, 'total_tokens': 258, 'completion_time': 0.145594933, 'completion_tokens_details': {'reasoning_tokens': 19}, 'prompt_time': 0.056513875, 'prompt_tokens_details': None, 'queue_time': 0.403129024, 'total_time': 0.202108808}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_bb691ea66b', 'service_tier'

In [47]:
from langgraph.types import Command
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware


agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": [
                        "approve",
                        "edit",
                        "reject"
                    ]
                },
                "read_email_tool": False,
            }
        )
    ],
)


config = {
    "configurable": {
        "thread_id": "email-thread-1"
    }
}


# ---------------------------
# 1. Start the agent
# ---------------------------

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Send an email to test@example.com "
                    "with subject 'Meeting' "
                    "and body 'Let's meet tomorrow at 10 AM.'"
                )
            }
        ]
    },
    config=config
)


# ---------------------------
# 2. Check for HITL interrupt
# ---------------------------

if "__interrupt__" in result:

    print("Paused for human intervention")

    import pprint
    pprint.pp(result["__interrupt__"])


    # ---------------------------
    # 3. Human approves
    # ---------------------------

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "approve"
                    }
                ]
            }
        ),
        config=config
    )


    print("Resumed after human intervention")
    print(result)

Paused for human intervention
[Interrupt(value={'action_requests': [{'name': 'send_email_tool',
                                       'args': {'body': "Let's meet tomorrow "
                                                        'at 10 AM.',
                                                'email_id': 'test@example.com',
                                                'subject': 'Meeting'},
                                       'description': 'Tool execution requires '
                                                      'approval\n'
                                                      '\n'
                                                      'Tool: send_email_tool\n'
                                                      "Args: {'body': "
                                                      '"Let\'s meet tomorrow '
                                                      'at 10 AM.", '
                                                      "'email_id': "
                             

In [63]:
### Editing
import uuid

config = {
    "configurable" : {"thread_id" : f"test-edit-{uuid.uuid4()}"}
}

result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello ")]},
    config=config
)

In [64]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello ", additional_kwargs={}, response_metadata={}, id='53e57d04-db2e-4d7a-88fd-b3505f866586'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "The user wants to send an email to wrong@email.com with subject 'Test' and body 'Hello'. We have a function to send email. Use send_email_tool with email_id, subject, body.", 'tool_calls': [{'id': 'fc_d81b505b-d3c3-469c-9617-624f9e28fb4d', 'function': {'arguments': '{"body":"Hello","email_id":"wrong@email.com","subject":"Test"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 87, 'prompt_tokens': 181, 'total_tokens': 268, 'completion_time': 0.184063625, 'completion_tokens_details': {'reasoning_tokens': 42}, 'prompt_time': 0.006968663, 'prompt_tokens_details': None, 'queue_time': 0.316982434, 'total_time': 0.191032288}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprin

In [65]:
edited = False
max_turns = 5
turns = 0

while "__interrupt__" in result and turns < max_turns:
    turns += 1

    if not edited:
        print("Paused!! editing")
        decision = {
            "type": "edit",
            "edited_action": {
                "name": "send_email_tool",
                "args": {
                    "email_id": "correct@email.com",
                    "subject": "Corrected Subject",
                    "body": "This was edited by human before sending"
                }
            }
        }
        edited = True
    else:
        # The model sometimes re-attempts the original (unedited) tool call
        # after an edit succeeds. Reject any further attempt instead of
        # editing again, so we don't loop forever.
        print("Rejecting repeat attempt of the original tool call")
        decision = {
            "type": "reject",
            "message": "The corrected email has already been sent successfully. Do not retry this tool call."
        }

    result = agent.invoke(
        Command(resume={"decisions": [decision]}),
        config=config
    )

result

Paused!! editing
Rejecting repeat attempt of the original tool call


{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello ", additional_kwargs={}, response_metadata={}, id='53e57d04-db2e-4d7a-88fd-b3505f866586'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "The user wants to send an email to wrong@email.com with subject 'Test' and body 'Hello'. We have a function to send email. Use send_email_tool with email_id, subject, body.", 'tool_calls': [{'id': 'fc_d81b505b-d3c3-469c-9617-624f9e28fb4d', 'function': {'arguments': '{"body":"Hello","email_id":"wrong@email.com","subject":"Test"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 87, 'prompt_tokens': 181, 'total_tokens': 268, 'completion_time': 0.184063625, 'completion_tokens_details': {'reasoning_tokens': 42}, 'prompt_time': 0.006968663, 'prompt_tokens_details': None, 'queue_time': 0.316982434, 'total_time': 0.191032288}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprin